# 02 — Tokenizer


**V1:**

In [1]:
import os
if not os.path.exists('MiniGPT'):
    !git clone https://github.com/userKk1/MiniGPT.git
%cd MiniGPT

Cloning into 'MiniGPT'...
remote: Enumerating objects: 109, done.
remote: Counting objects: 100% (109/109), done.
remote: Compressing objects: 100% (79/79), done.
remote: Total 109 (delta 48), reused 55 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (109/109), 168.47 KiB | 2.72 MiB/s, done.
Resolving deltas: 100% (48/48), done.
/content/MiniGPT


In [2]:
!python -m src.data

README.md: 100% 1.30k/1.30k [00:00<00:00, 2.85MB/s]
Resolving data files: 100% 54/54 [00:00<00:00, 19948.25it/s]
Collected 4138 files, 20.00 MB
Wrote /content/MiniGPT/data/raw/corpus_raw.txt (21.0 MB)


In [3]:
import sys
sys.path.append('.')

from config import cfg, DATA_RAW_DIR, DATA_PROCESSED_DIR
import json

corpus_path = DATA_RAW_DIR / 'corpus_raw.txt'
text = corpus_path.read_text(encoding='utf-8')
print(f'Corpus length: {len(text):,} characters')

Corpus length: 20,964,485 characters


## 1. Build the vocabulary

Every unique character in the corpus becomes one token.

In [4]:
chars = sorted(set(text))
vocab_size = len(chars)
print(f'Vocab size: {vocab_size}')
print(''.join(chars))

Vocab size: 99
	
 !"#$%&'()*+,-./0123456789:;<=>?@ABCDEFGHIJKLMNOPQRSTUVWXYZ[\]^_`abcdefghijklmnopqrstuvwxyz{|}~


## 2. Build encode/decode mappings

`stoi` (string-to-int) and `itos` (int-to-string)

In [5]:
stoi={ch:i for i,ch in enumerate(chars)}
itos={i:ch for i,ch in enumerate(chars)}

def encode(s:str)->list[int]:
    return [stoi[c] for c in s]

def decode(ids:list[int])->str:
    return ''.join(itos[i] for i in ids)


## 3. Sanity check: round-trip

In [6]:
sample = text[:200]
encoded = encode(sample)
decoded = decode(encoded)

print('Original :', repr(sample[:80]))
print('Decoded  :', repr(decoded[:80]))
assert decoded == sample, 'Round-trip failed — tokenizer is not lossless!'
print('Round-trip OK')

Original : '#!/usr/bin/env python\n# vim: tabstop=4 shiftwidth=4 softtabstop=4\n#\n# Copyright '
Decoded  : '#!/usr/bin/env python\n# vim: tabstop=4 shiftwidth=4 softtabstop=4\n#\n# Copyright '
Round-trip OK


## 4. Encode the full corpus and split train/val

In [7]:
import numpy as np

ids = encode(text)
data_arr = np.array(ids, dtype=np.uint16)  # fine as long as vocab_size < 65536

n = len(data_arr)

In [8]:
split_idx = int(n * cfg.data.train_split)
train_ids = data_arr[:split_idx]
val_ids = data_arr[split_idx:]

print(f'train: {len(train_ids):,} tokens')
print(f'val:   {len(val_ids):,} tokens')

train: 18,868,036 tokens
val:   2,096,449 tokens


**V2:BPE**

In [9]:
!pip install tokenizers

In [26]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel as ByteLevelPre
from tokenizers.decoders import ByteLevel as ByteLevelDecoder

In [11]:
corpus_path = DATA_RAW_DIR / "corpus_raw.txt"

text = corpus_path.read_text(encoding="utf-8")

print(len(text))

20964485


In [12]:
TOKENIZER_PATH = DATA_PROCESSED_DIR / "tokenizer.json"
VOCAB_SIZE=1000

In [35]:
tokenizer = Tokenizer(BPE(unk_token="<UNK>"))

tokenizer.pre_tokenizer = ByteLevelPre(add_prefix_space=False)

tokenizer.decoder = ByteLevelDecoder()

trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    special_tokens=["<UNK>"],
)

tokenizer.train(
    [str(corpus_path)],
    trainer
)

tokenizer.save(str(TOKENIZER_PATH))

print("Tokenizer trained.")
print("Saved to:", TOKENIZER_PATH)

print("Vocabulary size:", tokenizer.get_vocab_size())

Tokenizer trained.
Saved to: /content/MiniGPT/data/processed/tokenizer.json
Vocabulary size: 1000


In [36]:
vocab = tokenizer.get_vocab()

for token, token_id in list(vocab.items())[:100]:
    print(token_id, repr(token))

686 'rig'
799 'ren'
199 'and'
855 'Ġcan'
337 'rint'
582 'format'
692 '(['
128 'ion'
557 'Ġra'
792 '].'
521 '01'
158 'li'
581 'stat'
643 'join'
834 '."""'
80 'p'
746 'TI'
365 'alse'
78 'n'
610 'Field'
762 'write'
868 'da'
777 'right'
357 'sp'
935 'Ġobject'
912 'pected'
141 'mp'
125 'Ġi'
190 'mport'
533 'None'
263 'Ġ0'
936 'Ġ!'
130 'it'
506 'Ġpar'
374 'Ġ_'
258 '("'
828 'tings'
123 'ar'
16 '0'
388 'url'
937 'Ġz'
810 'state'
321 'Ġ=='
420 'bo'
638 '"]'
893 'core'
309 'ption'
621 'label'
410 'ĠM'
58 'Z'
245 'ype'
519 'atch'
930 'gr'
350 'der'
74 'j'
634 'Ġat'
287 'ĉĉ'
682 'dict'
98 'č'
395 'ers'
436 'ĠO'
292 'ĠF'
601 'ind'
615 'Ġro'
737 'server'
977 'lement'
127 'he'
965 'group'
425 'Ġprint'
522 'Ġdo'
148 'Ġs'
65 'a'
961 'Ġper'
650 'Ġal'
253 'arg'
507 'Ġlog'
920 'output'
152 'Ġin'
608 'Ġby'
136 '##'
576 'lient'
221 'ol'
849 'play'
413 'quest'
43 'K'
144 'ce'
447 'ise'
94 '~'
952 'input'
654 'Ġpy'
700 'thon'
754 'lock'
963 'Ġbo'
836 'client'
909 'method'
630 "Ġ{'"
402 'ĠG'
598 'ings'
902 'Ġ\

In [37]:
sample = """def __init__(self, value):
    self.value = value
    return self.value
"""

encoded = tokenizer.encode(sample)

print("TOKENS:")
print(encoded.tokens)

print("\nDECODED:")
decoded = tokenizer.decode(encoded.ids)
print(decoded)

print("\nIDENTICAL?")
print(sample == decoded)

TOKENS:
['def', 'Ġ__', 'init', '__(', 'self', ',', 'Ġvalue', '):', 'Ċ', 'ĠĠĠ', 'Ġself', '.', 'value', 'Ġ=', 'Ġvalue', 'Ċ', 'ĠĠĠ', 'Ġreturn', 'Ġself', '.', 'value', 'Ċ']

DECODED:
def __init__(self, value):
    self.value = value
    return self.value


IDENTICAL?
True


In [39]:
print(repr(sample))
print(repr(decoded))

'def __init__(self, value):\n    self.value = value\n    return self.value\n'
'def __init__(self, value):\n    self.value = value\n    return self.value\n'


In [40]:
print("Characters:", len(sample))
print("BPE tokens:", len(encoded.ids))
print("Compression:", len(sample) / len(encoded.ids))

Characters: 72
BPE tokens: 22
Compression: 3.272727272727273
